# AEON Credit Service Malaysia (ACSM) — Serverless Seamless Data Ingestion into BigQuery
**Run directly inside the BigQuery Studio UI (Colab Enterprise Notebook)**

This notebook demonstrates **100% Serverless, Zero-Service-Provisioning Data Ingestion** into BigQuery in **Singapore (`asia-southeast1`)**:
1. **Step 1**: Set parameterized `PROJECT_ID` (`asia-southeast1` Singapore) & clone the repository.
2. **Step 2**: Create the regional Cloud Storage landing bucket (`gs://acsm-workshop-landing-${PROJECT_ID}`) in **Singapore (`asia-southeast1`)**.
3. **Step 3**: Copy the 8 compressed `.csv.gz` dataset files (`69 MB` / `1,398,284` rows) to the Singapore bucket.
4. **Step 4**: Execute the `CREATE OR REPLACE TABLE` DDL SQL (`00_create_8_tables_ddl_with_descriptions.sql`) to create all **8 tables** (`Fact_EP_Judge` .. `dimProduct`) along with their table and column descriptions.
5. **Step 5**: Execute the Serverless `LOAD DATA OVERWRITE` SQL (`01_load_data_from_gcs.sql`) at **$0 Compute Cost (`0 Bytes Billed`)**.

---
## Step 0 (Prerequisite): Create VPC Network & Singapore (`asia-southeast1`) Subnetwork in Google Cloud Shell First

> **⚠️ IMPORTANT — RUN ONCE IN GOOGLE CLOUD SHELL BEFORE CONNECTING THIS NOTEBOOK**
> BigQuery Studio Notebooks (powered by Colab Enterprise) require a **VPC Network** and a **Regional Subnetwork in Singapore (`asia-southeast1`)** with **Private Google Access** enabled to provision the notebook runtime.
> 1. Click **Activate Cloud Shell (`>_`)** in the top-right corner of the Google Cloud Console.
> 2. Copy and run the `gcloud` commands below in **Cloud Shell**.
> 3. Once complete, come back to this Notebook in **BigQuery Studio**, click **Connect** (top-right), and select Network **`acsm-colab-network`** and Subnetwork **`acsm-colab-subnet-sg`** (`asia-southeast1`).

```bash
# =============================================================================
# Run these commands in Google Cloud Shell (>_) BEFORE connecting the notebook
# =============================================================================
export PROJECT_ID="<YOUR_GCP_PROJECT_ID>"   # e.g., export PROJECT_ID="trustedtesterarvind"
export LOCATION="asia-southeast1"           # Always Singapore (asia-southeast1)
export NETWORK_NAME="acsm-colab-network"
export SUBNET_NAME="acsm-colab-subnet-sg"

gcloud config set project "${PROJECT_ID}"

# 1. Enable required APIs for BigQuery Studio Notebooks (Colab Enterprise)
gcloud services enable \
  bigquery.googleapis.com \
  aiplatform.googleapis.com \
  compute.googleapis.com \
  dataform.googleapis.com \
  --project="${PROJECT_ID}"

# 2. Create Custom VPC Network
gcloud compute networks create "${NETWORK_NAME}" \
  --project="${PROJECT_ID}" \
  --subnet-mode=custom

# 3. Create Regional Subnetwork in Singapore (asia-southeast1) with Private Google Access
gcloud compute networks subnets create "${SUBNET_NAME}" \
  --project="${PROJECT_ID}" \
  --network="${NETWORK_NAME}" \
  --region="${LOCATION}" \
  --range="10.10.0.0/24" \
  --enable-private-ip-google-access

# 4. Create Cloud Router & Cloud NAT in Singapore (allows git clone from GitHub without public IPs)
gcloud compute routers create "acsm-colab-router-sg" \
  --project="${PROJECT_ID}" \
  --network="${NETWORK_NAME}" \
  --region="${LOCATION}"

gcloud compute routers nats create "acsm-colab-nat-sg" \
  --project="${PROJECT_ID}" \
  --router="acsm-colab-router-sg" \
  --region="${LOCATION}" \
  --auto-allocate-nat-external-ips \
  --nat-all-subnet-ip-ranges
```

> **🔍 How to Verify Step 0 on GCP Console UI & Connect the Notebook Runtime**
> 1. Open **VPC network $\rightarrow$ VPC networks** in the GCP Console and verify **`acsm-colab-network`** and subnet **`acsm-colab-subnet-sg`** (`Region: asia-southeast1`, `Private Google access: On`) are listed.
> 2. Come back to **BigQuery Studio**, open this notebook, click the **Connect** dropdown (top-right) $\rightarrow$ **Connect to a runtime** (or **Create a runtime template** in `asia-southeast1`), select Network **`acsm-colab-network`** and Subnetwork **`acsm-colab-subnet-sg`**, and click **Connect**.

---
## Step 1: Configure Parameters (`PROJECT_ID` & Singapore Region) and Clone Repository

In [ ]:
# @title Step 1: Set Parameters & Clone Repository { display-mode: "form" }
PROJECT_ID = "decoded-effect-509506-m5"  # @param {type:"string"}
LOCATION = "asia-southeast1"             # @param ["asia-southeast1"]
BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"

%load_ext google.cloud.bigquery
!git clone https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git 2>/dev/null || git -C aeon-credit-gcp-workshop pull origin main
!gcloud config set project $PROJECT_ID

> **🔍 How to Verify Step 1 on GCP Console UI**
> 1. In the top Google Cloud Console bar, confirm your **`PROJECT_ID`** is selected in the Project Picker.
> 2. In the cell output above, confirm the repository cloned cleanly and `LOCATION=asia-southeast1` (Singapore) is active.
---
## Step 2: Create the Cloud Storage Landing Bucket in Singapore (`asia-southeast1`)

In [ ]:
# @title Step 2: Create Regional GCS Bucket in Singapore (asia-southeast1)
!gcloud storage buckets describe gs://{BUCKET_NAME} --project={PROJECT_ID} >/dev/null 2>&1 || \
  gcloud storage buckets create gs://{BUCKET_NAME} \
    --project={PROJECT_ID} \
    --location={LOCATION} \
    --uniform-bucket-level-access

!gcloud storage buckets describe gs://{BUCKET_NAME} --format="table(name,location,location_type,storage_class)"

> **🔍 How to Verify Step 2 on GCP Console UI (Cloud Storage Browser)**
> 1. Open **Cloud Storage $\rightarrow$ Buckets** in the GCP Console.
> 2. Verify bucket **`acsm-workshop-landing-${PROJECT_ID}`** shows **Location type**: `Region`, **Location**: `asia-southeast1 (Singapore)`, and **Public access**: `Not public`.
---
## Step 3: Copy Compressed Dataset Files (`.csv.gz`) from Repo to Singapore Bucket

In [ ]:
# @title Step 3: Upload All 8 Compressed .csv.gz Files (69 MB / 1.4M Rows) to GCS
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/
!gcloud storage ls -l gs://{BUCKET_NAME}/full_compressed/

> **🔍 How to Verify Step 3 on GCP Console UI (Bucket Objects View)**
> 1. In **Cloud Storage $\rightarrow$ Buckets**, click **`acsm-workshop-landing-${PROJECT_ID}` $\rightarrow$ `full_compressed/`**.
> 2. Click **Refresh** and verify all `.csv.gz` files (`T1_Fact_EP_Judge.csv.gz` .. `T8_dimProduct.csv.gz`) are listed in `asia-southeast1 (Singapore)` with `application/gzip` content type.
---
## Step 4: Run `CREATE TABLE` DDL Statement (With All Table & 226 Column Descriptions)

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- =============================================================================
-- DEMO FLOW 2 (STEP 1 OF 2): Create 8 ACSM Tables with Table & Column Descriptions
-- Project: `<PROJECT_ID>` (e.g. `trustedtesterarvind`) | Dataset: `acsm_bronze`
-- Location: `asia-southeast1` (Singapore)
-- Source Dictionary: `Mock Metadata.xlsx` (8 tables, 226 described columns)
-- =============================================================================

CREATE SCHEMA IF NOT EXISTS `acsm_bronze`
OPTIONS (
  location = "asia-southeast1",
  description = "AEON Credit Service Malaysia (ACSM) — 8 Core Tables (T1-T8) with Governed Metadata (Singapore Region)"
);

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Judge` (
  `Rcd_DT` DATE OPTIONS(description = "Data extraction date [Source Data Type: date]"),
  `CIF_NO` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: numeric]"),
  `APPL_NO` INT64 OPTIONS(description = "PRODUCT application number [Source Data Type: varchar]"),
  `AGREE_NO` INT64 OPTIONS(description = "PRODUCT loan account number [Source Data Type: numeric]"),
  `APPL_DT` INT64 OPTIONS(description = "Easy payment application date [Source Data Type: numeric]"),
  `APPL_STS` STRING OPTIONS(description = "Easy payment application status [Source Data Type: varchar]"),
  `JUDGE_DT` INT64 OPTIONS(description = "Easy payment application decision date [Source Data Type: numeric]"),
  `SCORING_POINT` INT64 OPTIONS(description = "Easy payment Final Score for decision [Source Data Type: numeric]"),
  `SCORING_TYPE` INT64 OPTIONS(description = "Easy payment Final Score type [Source Data Type: numeric]"),
  `SCORING_RANK` STRING OPTIONS(description = "Easy payment Final score rank based on bucket [Source Data Type: varchar]"),
  `LOAN_CODE` INT64 OPTIONS(description = "Loan type [Source Data Type: varchar]"),
  `LOAN_TYP_ID` INT64 OPTIONS(description = "Loan subtype [Source Data Type: varchar]"),
  `LOAN_GRP` INT64 OPTIONS(description = "Loan group [Source Data Type: varchar]"),
  `AGENT_CODE1` INT64 OPTIONS(description = "Merchant group [Source Data Type: varchar]"),
  `AGENT_CODE2` INT64 OPTIONS(description = "Merchant subgroup [Source Data Type: varchar]"),
  `APPL_CHANNEL` STRING OPTIONS(description = "Application channel [Source Data Type: varchar]"),
  `REJECT_CODE` STRING OPTIONS(description = "Reject Code [Source Data Type: varchar]"),
  `FIN_AMT` FLOAT64 OPTIONS(description = "Easy payment finance approved amount [Source Data Type: numeric]"),
  `FIN_PRFT_AMT` FLOAT64 OPTIONS(description = "Easy payment profit/interest amount [Source Data Type: numeric]"),
  `FIN_TOTAL_AMT` FLOAT64 OPTIONS(description = "Easy payment financing total amount [Source Data Type: numeric]"),
  `INST_AMT` FLOAT64 OPTIONS(description = "Easy payment installment amount [Source Data Type: numeric]"),
  `DEPOSIT` INT64 OPTIONS(description = "% for downpayment [Source Data Type: numeric]"),
  `INTEREST` FLOAT64 OPTIONS(description = "Easy payment interest [Source Data Type: decimal]"),
  `TOTAL_INST` INT64 OPTIONS(description = "Easy payment total installment months; loan tenure [Source Data Type: numeric]"),
  `JointIncome_FG` STRING OPTIONS(description = "Joint income yes/no flag [Source Data Type: varchar]"),
  `SCORE_DECISION` STRING OPTIONS(description = "Easy payment score decision: accept/decline [Source Data Type: char]"),
  `NetIncome` FLOAT64 OPTIONS(description = "Easy payment applicant net income [Source Data Type: numeric]"),
  `Income` FLOAT64 OPTIONS(description = "Easy payment applicant income [Source Data Type: numeric]"),
  `Age` INT64 OPTIONS(description = "Easy payment applicant age [Source Data Type: int]"),
  `HomeYear` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: numeric]"),
  `YearOfBusiness` INT64 OPTIONS(description = "Number of years of employment in current company [Source Data Type: numeric]"),
  `AnnualIncome` FLOAT64 OPTIONS(description = "Annual income [Source Data Type: numeric]"),
  `TOTAL_AEON_INST` INT64 OPTIONS(description = "Total installment for all Aeon products [Source Data Type: numeric]"),
  `TOTAL_AEON_OSB` FLOAT64 OPTIONS(description = "Total outstanding balance for all Aeon products [Source Data Type: numeric]"),
  `CUR_REPAY_RATIO` FLOAT64 OPTIONS(description = "Current repayment ratio [Source Data Type: decimal]"),
  `NEW_REPAY_RATIO` FLOAT64 OPTIONS(description = "New repayment ratio [Source Data Type: decimal]"),
  `NDI` FLOAT64 OPTIONS(description = "Net disposable income [Source Data Type: numeric]"),
  `CUR_DSR` FLOAT64 OPTIONS(description = "Current debt to service ratio [Source Data Type: decimal]"),
  `NEW_DSR` FLOAT64 OPTIONS(description = "New debt to service ratio [Source Data Type: decimal]"),
  `B_OtherIncome` FLOAT64 OPTIONS(description = "Other income [Source Data Type: numeric]"),
  `B_NonBankCommitment` FLOAT64 OPTIONS(description = "Non-bank commitment [Source Data Type: numeric]"),
  `DEPENDANT` INT64 OPTIONS(description = "Children, spouse. [Source Data Type: numeric]"),
  `YEAR_MADE` INT64 OPTIONS(description = "Easy payment vehicle year made [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of Business [Source Data Type: varchar]"),
  `HomeOwner_flag` BOOL OPTIONS(description = "Ownership check of home [Source Data Type: varchar]"),
  `CIFState` STRING OPTIONS(description = "Customer address state as at application [Source Data Type: varchar]"),
  `Race` STRING OPTIONS(description = "Customer race as at application [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Customer gender as at application [Source Data Type: varchar]"),
  `Marital` STRING OPTIONS(description = "Customer marital status as at application [Source Data Type: varchar]"),
  `National` STRING OPTIONS(description = "Customer nationality as at application [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Customer occupation as at application [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Customer academic qualification as at application [Source Data Type: varchar]"),
  `B_AdvInstallPymt` FLOAT64 OPTIONS(description = "Easy payment advanced install payment [Source Data Type: numeric]"),
  `AdvInstallPymtBand` STRING OPTIONS(description = "Easy payment advanced installment payment band [Source Data Type: nvarchar]"),
  `JudgeWeek_FG` INT64 OPTIONS(description = "Judge week flag [Source Data Type: varchar]"),
  `Biometric_FG` BOOL OPTIONS(description = "Biometric indicator [Source Data Type: varchar]"),
  `E-KYC` STRING OPTIONS(description = "E-KYC pass/fail [Source Data Type: varchar]"),
  `OTP` STRING OPTIONS(description = "OTP pass/fail [Source Data Type: varchar]"),
  `APPLY_FIN_AMT` FLOAT64 OPTIONS(description = "Easy payment applied financing amount [Source Data Type: numeric]"),
  `PreAssessment_Flag` BOOL OPTIONS(description = "Some customers qualify for pre-assessment [Source Data Type: varchar]")
) OPTIONS(description = "The current (daily full refreshed) application status (Approved or Rejected) for EP products. (Governed via Mock Metadata.xlsx | Sheet: T1 - Fact_EP_Judge)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Sales` (
  `TX_DT` FLOAT64 OPTIONS(description = "Sales Date"),
  `CIF_No` STRING OPTIONS(description = "Unique customer ID"),
  `TransactionCountry` STRING OPTIONS(description = "Ringgit Malaysia"),
  `TransactionCountryHigherLevel` STRING OPTIONS(description = "Local or Oversea"),
  `PriviledgeMerchantsGrp` STRING OPTIONS(description = "Member merchant"),
  `LDESC` STRING OPTIONS(description = "Spend Location"),
  `Sales_Type` STRING OPTIONS(description = "Cash Purchase or Cash Advance"),
  `Amount` FLOAT64 OPTIONS(description = "Transaction Amount"),
  `TransCount` FLOAT64 OPTIONS(description = "Transaction Count")
) OPTIONS(description = "The confirmed sales log of the EP product. (Governed via Mock Metadata.xlsx | Sheet: T2 - Fact_EP_Sales)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Collection` (
  `TX_DT` INT64 OPTIONS(description = "Reporting date [Source Data Type: decimal]"),
  `First_INST_DT` INT64 OPTIONS(description = "First intallment date [Source Data Type: decimal]"),
  `Agree_No` INT64 OPTIONS(description = "Loan agreement ID [Source Data Type: varchar]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `Current_Time_Payment` INT64 OPTIONS(description = "current installment period [Source Data Type: varchar]"),
  `Del_Sts` INT64 OPTIONS(description = "Lock Deliquency status [Source Data Type: varchar]"),
  `Collection_Branch` INT64 OPTIONS(description = "Branch ID [Source Data Type: varchar]"),
  `Score_Value` INT64 OPTIONS(description = "Collection Score Point [Source Data Type: decimal]"),
  `Score_Grade` STRING OPTIONS(description = "Collection Score Grade [Source Data Type: varchar]"),
  `Sub_Code` STRING OPTIONS(description = "AKPK checker [Source Data Type: varchar]"),
  `Pay_in_Full` INT64 OPTIONS(description = "Account Status [Source Data Type: varchar]"),
  `Classification_Code` STRING OPTIONS(description = "Life Deliquency status [Source Data Type: varchar]"),
  `FinPlus_Code` STRING OPTIONS(description = "FinPlus (e-credit evaluation) tier [Source Data Type: varchar]"),
  `MDD` INT64 OPTIONS(description = "EP Multi Due Date [Source Data Type: varchar]"),
  `LoanTyp_ID` INT64 OPTIONS(description = "Loan product type indicator [Source Data Type: varchar]"),
  `Billing_OSP` FLOAT64 OPTIONS(description = "Principal Billing amount [Source Data Type: decimal]"),
  `Billing_Count` INT64 OPTIONS(description = "Principal Billing count [Source Data Type: decimal]"),
  `Collection_OSP` FLOAT64 OPTIONS(description = "Principal Collection amount [Source Data Type: decimal]"),
  `Collection_Count` INT64 OPTIONS(description = "Principal Collection count [Source Data Type: decimal]"),
  `Unpaid_OSP` FLOAT64 OPTIONS(description = "Principal Unpaid amount [Source Data Type: decimal]"),
  `Unpaid_Count` INT64 OPTIONS(description = "Principal Unpaid count [Source Data Type: decimal]")
) OPTIONS(description = "The collection status snapshot for EP products as at closing period (Governed via Mock Metadata.xlsx | Sheet: T3 - Fact_EP_Collection)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Judge` (
  `Rcd_DT` DATE OPTIONS(description = "Data extraction date [Source Data Type: date]"),
  `Account_No` INT64 OPTIONS(description = "Card account number [Source Data Type: varchar]"),
  `Appl_ID` INT64 OPTIONS(description = "Credit card application ID [Source Data Type: varchar]"),
  `ApplSts_ID` STRING OPTIONS(description = "Credit card application status ID [Source Data Type: varchar]"),
  `CIF_ID` INT64 OPTIONS(description = "Credit card customer ID [Source Data Type: varchar]"),
  `CardTyp_ID` STRING OPTIONS(description = "Credit card type [Source Data Type: varchar]"),
  `CardBrand_ID` STRING OPTIONS(description = "Credit card brand [Source Data Type: varchar]"),
  `CardApplTyp_ID` STRING OPTIONS(description = "Principal/supplementary [Source Data Type: varchar]"),
  `ApplChnnl_ID` STRING OPTIONS(description = "Credit card application channel [Source Data Type: varchar]"),
  `Reject_ID` INT64 OPTIONS(description = "Credit card rejection reason ID [Source Data Type: varchar]"),
  `Decline_ID` INT64 OPTIONS(description = "Rejection reason [Source Data Type: varchar]"),
  `Agent_ID` INT64 OPTIONS(description = "Merchant ID [Source Data Type: varchar]"),
  `ScoreDecision_ID` STRING OPTIONS(description = "Credit card score decision category [Source Data Type: varchar]"),
  `ScoreRank_ID` STRING OPTIONS(description = "Credit card score rank [Source Data Type: varchar]"),
  `SysRcmmd_ID` STRING OPTIONS(description = "System recommended decision [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Gender [Source Data Type: varchar]"),
  `Age` INT64 OPTIONS(description = "Age [Source Data Type: int]"),
  `Race` STRING OPTIONS(description = "Race [Source Data Type: varchar]"),
  `Nationality` STRING OPTIONS(description = "Nationality short code [Source Data Type: varchar]"),
  `MaritalSts` STRING OPTIONS(description = "Marital Status [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Highest academic qualification [Source Data Type: varchar]"),
  `YrStay` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: int]"),
  `HomeOwn` STRING OPTIONS(description = "Type of home ownership [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Occupation [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of business of customers employer [Source Data Type: varchar]"),
  `YrJob` INT64 OPTIONS(description = "Year in job [Source Data Type: int]"),
  `NetIncome` INT64 OPTIONS(description = "Net income [Source Data Type: int]"),
  `GrossIncome` INT64 OPTIONS(description = "Gross income [Source Data Type: int]"),
  `AnnualIncome` INT64 OPTIONS(description = "Annual income [Source Data Type: int]"),
  `AnnualIncomeTotLmt` INT64 OPTIONS(description = "Annual income total limit [Source Data Type: int]"),
  `CurrRepay` INT64 OPTIONS(description = "Current repayment amount [Source Data Type: int]"),
  `NewRepay` INT64 OPTIONS(description = "New repayment amount [Source Data Type: int]"),
  `RcmmdIntrst` FLOAT64 OPTIONS(description = "Recommended interest rate [Source Data Type: decimal]"),
  `NDI` INT64 OPTIONS(description = "Net Disposable Income [Source Data Type: int]"),
  `CurrDSR` INT64 OPTIONS(description = "Current DSR [Source Data Type: decimal]"),
  `NewDSR` INT64 OPTIONS(description = "New DSR [Source Data Type: decimal]"),
  `PaySlipTyp_ID` INT64 OPTIONS(description = "Payslip type [Source Data Type: varchar]"),
  `CardActivate_FG` BOOL OPTIONS(description = "Card activated [Source Data Type: varchar]"),
  `EmergencyCont_FG` BOOL OPTIONS(description = "Emergency contact provided [Source Data Type: varchar]"),
  `CardActivate_DT` INT64 OPTIONS(description = "Card activation date [Source Data Type: decimal]"),
  `Judge_DT` INT64 OPTIONS(description = "Decision date [Source Data Type: decimal]"),
  `Appl_DT` INT64 OPTIONS(description = "Application date [Source Data Type: decimal]"),
  `B_Limit` FLOAT64 OPTIONS(description = "Total limit [Source Data Type: decimal]"),
  `B_CrLimit` INT64 OPTIONS(description = "Credit limit [Source Data Type: decimal]"),
  `B_CashAdvLimit` FLOAT64 OPTIONS(description = "Cash advance limit [Source Data Type: decimal]"),
  `B_NonBankCommitment` FLOAT64 OPTIONS(description = "Non bank commitment [Source Data Type: decimal]"),
  `CIFState_ID` STRING OPTIONS(description = "State [Source Data Type: varchar]"),
  `Biometric_FG` STRING OPTIONS(description = "Biometric indicator [Source Data Type: varchar]"),
  `FinalDecline_ID` INT64 OPTIONS(description = "Decline ID [Source Data Type: varchar]"),
  `Final_Score` INT64 OPTIONS(description = "Credit card CTOS score [Source Data Type: decimal]"),
  `Final_Score_Type` INT64 OPTIONS(description = "Credit card CTOS score [Source Data Type: decimal]"),
  `Final_ScoreDesc` STRING OPTIONS(description = "Credit card CTOS score description [Source Data Type: varchar]"),
  `ApplyCardBiz_ID` STRING OPTIONS(description = "Applied card ID [Source Data Type: varchar]"),
  `ProposedCardBiz_ID` STRING OPTIONS(description = "Proposed card ID [Source Data Type: varchar]")
) OPTIONS(description = "The current (daily full refreshed) application status (Approved or Rejected) for credit cards. (Governed via Mock Metadata.xlsx | Sheet: T4 - Fact_CC_Judge)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Sales` (
  `TX_DT` INT64 OPTIONS(description = "Sales Date [Source Data Type: decimal]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `TransactionCountry` STRING OPTIONS(description = "Ringgit Malaysia [Source Data Type: varchar]"),
  `TransactionCountryHigherLevel` STRING OPTIONS(description = "Local or Oversea [Source Data Type: varchar]"),
  `PriviledgeMerchantsGrp` STRING OPTIONS(description = "Member merchant [Source Data Type: varchar]"),
  `LDESC` STRING OPTIONS(description = "Spend Location [Source Data Type: varchar]"),
  `Sales_Type` STRING OPTIONS(description = "Cash Purchase or Cash Advance [Source Data Type: varchar]"),
  `Amount` FLOAT64 OPTIONS(description = "Transaction Amount [Source Data Type: decimal]"),
  `TransCount` INT64 OPTIONS(description = "Transaction Count [Source Data Type: decimal]")
) OPTIONS(description = "The actual spending & cash advance log on credit card (Governed via Mock Metadata.xlsx | Sheet: T5 - Fact_CC_Sales)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Collection` (
  `TX_DT` INT64 OPTIONS(description = "Reporting date [Source Data Type: decimal]"),
  `Account_No` INT64 OPTIONS(description = "CC account ID [Source Data Type: varchar]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `DC_Sts` INT64 OPTIONS(description = "Lock deliquency status [Source Data Type: varchar]"),
  `Application_Branch` INT64 OPTIONS(description = "Branch ID [Source Data Type: varchar]"),
  `Score_Value` INT64 OPTIONS(description = "Collection Score point [Source Data Type: decimal]"),
  `Score_Grade` STRING OPTIONS(description = "Collection Score grade [Source Data Type: varchar]"),
  `MDD` INT64 OPTIONS(description = "CC Due Date [Source Data Type: varchar]"),
  `FinPlus_Code` STRING OPTIONS(description = "FinPlus (e-credit evaluation) tier [Source Data Type: varchar]"),
  `Billing_OSP` FLOAT64 OPTIONS(description = "Billing amount [Source Data Type: decimal]"),
  `Billing_Count` INT64 OPTIONS(description = "Billing count [Source Data Type: decimal]"),
  `Collection_OSP` FLOAT64 OPTIONS(description = "Collection amount [Source Data Type: decimal]"),
  `Collection_Count` INT64 OPTIONS(description = "Collection count [Source Data Type: decimal]"),
  `Unpaid_OSP` FLOAT64 OPTIONS(description = "Unpaid amount [Source Data Type: decimal]"),
  `Unpaid_Count` INT64 OPTIONS(description = "Unpaid count [Source Data Type: decimal]"),
  `Maintain_OSP` FLOAT64 OPTIONS(description = "Maintain amount [Source Data Type: decimal]"),
  `Maintain_Count` INT64 OPTIONS(description = "Maintain count [Source Data Type: decimal]")
) OPTIONS(description = "The billing and collection status snapshot for credit cards as at closing period (Governed via Mock Metadata.xlsx | Sheet: T6 - Fact_CC_Collection)");

CREATE OR REPLACE TABLE `acsm_bronze.m3CIF` (
  `Rcd_DT` DATE OPTIONS(description = "Record refresh date [Source Data Type: date]"),
  `CIF_ID` INT64 OPTIONS(description = "Customer ID [Source Data Type: varchar]"),
  `CIF_NM` STRING OPTIONS(description = "Customer name [Source Data Type: varchar]"),
  `CIF_NM1` STRING OPTIONS(description = "Customer name [Source Data Type: varchar]"),
  `CIF_NM2` STRING OPTIONS(description = "Customer name [Source Data Type: varchar]"),
  `MaritalSts` STRING OPTIONS(description = "MaritalStatus [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Gender [Source Data Type: varchar]"),
  `Citizen` STRING OPTIONS(description = "Citizen [Source Data Type: varchar]"),
  `State` STRING OPTIONS(description = "State [Source Data Type: varchar]"),
  `Region` STRING OPTIONS(description = "Region [Source Data Type: varchar]"),
  `Race` STRING OPTIONS(description = "Race [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of business [Source Data Type: varchar]"),
  `HomeOwn` STRING OPTIONS(description = "Customer's homeowner category [Source Data Type: varchar]"),
  `HomePost` INT64 OPTIONS(description = "Customer's home postcode [Source Data Type: varchar]"),
  `_HomeAddr1` STRING OPTIONS(description = "_HomeAddr1 [Source Data Type: varchar]"),
  `_HomeAddr2` STRING OPTIONS(description = "_HomeAddr2 [Source Data Type: varchar]"),
  `_HomeAddr3` STRING OPTIONS(description = "_HomeAddr3 [Source Data Type: varchar]"),
  `EmpPost` INT64 OPTIONS(description = "Employment Postal Code [Source Data Type: varchar]"),
  `MailPost` INT64 OPTIONS(description = "Mail Postal Code [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Occupation [Source Data Type: varchar]"),
  `PayslipTyp` INT64 OPTIONS(description = "PayslipTyp [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Academic qualifications [Source Data Type: varchar]"),
  `Emp_NM` STRING OPTIONS(description = "Employer Name [Source Data Type: varchar]"),
  `SelftEmp_FG` BOOL OPTIONS(description = "Self Employment Indicator [Source Data Type: varchar]"),
  `Felda_FG` BOOL OPTIONS(description = "Felda is a government program to help rural Malaysians. [Source Data Type: varchar]"),
  `JoinIncome_FG` BOOL OPTIONS(description = "Joint Income Indicator [Source Data Type: varchar]"),
  `RecvPromo_FG` BOOL OPTIONS(description = "Received promotion indicator [Source Data Type: varchar]"),
  `N_Age` INT64 OPTIONS(description = "Age [Source Data Type: numeric]"),
  `N_YrStay` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: numeric]"),
  `N_YrJob` INT64 OPTIONS(description = "Number of years in current job [Source Data Type: numeric]"),
  `B_NetIncome` FLOAT64 OPTIONS(description = "Net Income [Source Data Type: numeric]"),
  `B_GrossIncome` FLOAT64 OPTIONS(description = "Gross Income [Source Data Type: numeric]"),
  `B_AnnualIncome` FLOAT64 OPTIONS(description = "Annual Income [Source Data Type: numeric]"),
  `EmpSts` INT64 OPTIONS(description = "Employment Status [Source Data Type: int]")
) OPTIONS(description = "Customer latest status daily refresh table (Governed via Mock Metadata.xlsx | Sheet: T7 - m3CIF)");

CREATE OR REPLACE TABLE `acsm_bronze.dimProduct` (
  `Expiry_DT` INT64 OPTIONS(description = "Card expiry date [Source Data Type: varchar]"),
  `FirstSpend_DT` INT64 OPTIONS(description = "First spend date [Source Data Type: varchar]"),
  `Block_Code` STRING OPTIONS(description = "If card is blocked [Source Data Type: varchar]"),
  `Block_Date` INT64 OPTIONS(description = "If card is blocked [Source Data Type: numeric]"),
  `CIC_Status` STRING OPTIONS(description = "CIC status [Source Data Type: varchar]"),
  `Card_Status` STRING OPTIONS(description = "Card status [Source Data Type: varchar]"),
  `AKPK_Status` BOOL OPTIONS(description = "AKPK status [Source Data Type: varchar]"),
  `Card_First_Emboss_Date` INT64 OPTIONS(description = "Physical card issuance date [Source Data Type: numeric]"),
  `Card_Emboss_Date` INT64 OPTIONS(description = "Card emboss date [Source Data Type: numeric]"),
  `Card_First_Activated_Date` INT64 OPTIONS(description = "Card first activated date [Source Data Type: numeric]"),
  `Card_Activated_Date` INT64 OPTIONS(description = "Card current activated date [Source Data Type: numeric]"),
  `CP_CL` FLOAT64 OPTIONS(description = "Credit Purchase Total Limit [Source Data Type: numeric]"),
  `CP_CL_Available` FLOAT64 OPTIONS(description = "Credit Purchase Available Limit [Source Data Type: numeric]"),
  `CA_CL` FLOAT64 OPTIONS(description = "Credit Advance Total Limit [Source Data Type: numeric]"),
  `CA_CL_Available` FLOAT64 OPTIONS(description = "Credit Advance Available Limit [Source Data Type: numeric]"),
  `CP_CL_Usage` FLOAT64 OPTIONS(description = "Credit Purchase Usage [Source Data Type: numeric]"),
  `CA_CL_Usage` FLOAT64 OPTIONS(description = "Credit Advance Usage [Source Data Type: numeric]"),
  `CIF_ID` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `Account_No` INT64 OPTIONS(description = "Card account number [Source Data Type: numeric]"),
  `Account_Agree_Sts` STRING OPTIONS(description = "Account agreement status [Source Data Type: varchar]"),
  `Virtual_Card_Flag` BOOL OPTIONS(description = "Virtual card indicator [Source Data Type: varchar]"),
  `Wallet_Tier` STRING OPTIONS(description = "Loyalty account Tier [Source Data Type: varchar]")
) OPTIONS(description = "The current and daily full refresh master record of all active cards (Governed via Mock Metadata.xlsx | Sheet: T8 - dimProduct)");


> **🔍 How to Verify Step 4 on GCP Console UI (BigQuery Studio Explorer & Schema Tab)**
> 1. In the left **BigQuery Studio Explorer** pane (right beside this notebook!), expand **`${PROJECT_ID}` $\rightarrow$ `acsm_bronze`**.
> 2. Click on **`Fact_EP_Judge`** $\rightarrow$ select the **Schema** tab to verify all **60 columns** have their business **Description** populated from `Mock Metadata.xlsx`, and check the **Details** tab to confirm **Data location** is `asia-southeast1` and **Number of rows** is currently `0`.
---
## Step 5: Run Serverless `LOAD DATA OVERWRITE` Statement (**$0 Load Cost / `0 B Billed`**)
> **💡 Zero Compute Cost (`$0.00` / `0 Bytes Billed`)**: Batch loading data into BigQuery from Cloud Storage via the `LOAD DATA` SQL statement is **100% FREE (`$0.00`)** using BigQuery's shared batch slot pool ([BigQuery Pricing](https://cloud.google.com/bigquery/pricing#loading_data)). Because both the bucket and dataset are in **Singapore (`asia-southeast1`)**, there is **$0 network egress cost** and **100% of the 226 column descriptions** from Step 4 are preserved automatically.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- =============================================================================
-- DEMO FLOW 2 (STEP 2 OF 2): Serverless SQL `LOAD DATA OVERWRITE` from GCS
-- Loads all 1,398,284 records from `gs://acsm-workshop-landing-decoded-effect-509506-m5`
-- into the 8 pre-created tables while preserving all 226 column descriptions.
-- Zero compute provisioning required | $0 BigQuery batch load cost (0 B billed).
-- =============================================================================

-- Load Fact_EP_Judge
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Judge`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T1_Fact_EP_Judge.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load Fact_EP_Sales
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Sales`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T2_Fact_EP_Sales.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load Fact_EP_Collection
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Collection`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T3_Fact_EP_Collection.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load Fact_CC_Judge
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Judge`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T4_Fact_CC_Judge_v2.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load Fact_CC_Sales
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Sales`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T5_Fact_CC_Sales.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load Fact_CC_Collection
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Collection`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T6_Fact_CC_Collection.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load m3CIF
LOAD DATA OVERWRITE `acsm_bronze.m3CIF`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T7_m3CIF.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);

-- Load dimProduct
LOAD DATA OVERWRITE `acsm_bronze.dimProduct`
FROM FILES (
  format = 'CSV',
  uris = ['gs://acsm-workshop-landing-decoded-effect-509506-m5/full_compressed/T8_dimProduct.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);


---
## Step 6: Pure SQL Verification (`INFORMATION_SCHEMA` Audits — Zero Python)

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 1: Audit Row Counts, Table Descriptions & Described Columns (with Grand Total)
WITH table_stats AS (
  SELECT
    t.table_name,
    s.row_count,
    COUNT(c.column_name) AS total_columns,
    COUNTIF(c.description IS NOT NULL AND c.description != "") AS described_columns,
    REGEXP_REPLACE(COALESCE(opt.option_value, ""), r"^\"|\"$", "") AS table_description
  FROM `acsm_bronze.INFORMATION_SCHEMA.TABLES` t
  JOIN `acsm_bronze.__TABLES__` s
    ON t.table_name = s.table_id
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.TABLE_OPTIONS` opt
    ON t.table_name = opt.table_name AND opt.option_name = "description"
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS` c
    ON t.table_name = c.table_name
  WHERE t.table_name IN ("Fact_EP_Judge", "Fact_EP_Sales", "Fact_EP_Collection", "Fact_CC_Judge", "Fact_CC_Sales", "Fact_CC_Collection", "m3CIF", "dimProduct")
  GROUP BY 1, 2, 5
)
SELECT
  table_name,
  row_count,
  total_columns,
  described_columns,
  ROUND(SAFE_DIVIDE(described_columns, total_columns) * 100, 1) AS coverage_pct,
  table_description
FROM table_stats
UNION ALL
SELECT
  "TOTAL (ALL 8 ACSM TABLES)" AS table_name,
  SUM(row_count) AS row_count,
  SUM(total_columns) AS total_columns,
  SUM(described_columns) AS described_columns,
  ROUND(SAFE_DIVIDE(SUM(described_columns), SUM(total_columns)) * 100, 1) AS coverage_pct,
  "100% Serverless Load Complete | 0 Bytes Billed ($0.00)" AS table_description
FROM table_stats
ORDER BY CASE WHEN STARTS_WITH(table_name, "TOTAL") THEN 2 ELSE 1 END, table_name;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 2: Prove $0 Load Cost (0 Bytes Billed) via BigQuery INFORMATION_SCHEMA.JOBS
SELECT
  job_id,
  statement_type,
  destination_table.table_id AS loaded_table,
  state,
  COALESCE(total_bytes_billed, 0) AS bytes_billed,
  "$0.00 (Free Shared Batch Pool)" AS ingestion_compute_cost,
  TIMESTAMP_DIFF(end_time, start_time, SECOND) AS duration_seconds
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE statement_type = "LOAD_DATA"
  AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
ORDER BY creation_time DESC
LIMIT 8;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 3: Inspect Governed Column Descriptions from INFORMATION_SCHEMA
SELECT
  table_name,
  column_name,
  data_type,
  description
FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
WHERE table_name IN ("Fact_EP_Judge", "Fact_EP_Sales", "Fact_EP_Collection", "Fact_CC_Judge", "Fact_CC_Sales", "Fact_CC_Collection", "m3CIF", "dimProduct")
ORDER BY table_name, column_name
LIMIT 25;


> **🔍 How to Verify Step 5 on GCP Console UI (4 Visual Checks in BigQuery Studio)**
> 1. **Verify `$0` Load Cost (`0 B Billed`)**: In the cell output above (or in BigQuery **Job history**), point out **`Total Bytes Billed: 0 B ($0.00 FREE Serverless Batch Load)`** in **`asia-southeast1`**.
> 2. **Verify Row Counts (`Details` Tab)**: In the left Explorer tree, click **`Fact_CC_Sales`** ($535,925$ rows) or **`Fact_EP_Judge`** ($140,000$ rows) $\rightarrow$ **Details** tab.
> 3. **Verify Loaded Records (`Preview` Tab — also $0 Cost)**: Click the **Preview** tab on any table to browse the records at zero query cost (`0 B billed`).
> 4. **Verify Preserved Column Descriptions (`Schema` Tab)**: Click the **Schema** tab to confirm all **226 column descriptions** remained intact after `LOAD DATA OVERWRITE`.